# Lab: Sentiment Analysis  
#  *******Data-Centric vs Model-Centric approaches




This lab gives an introduction to sentiment analysis approaches.

In this lab, we'll build a classifier for product reviews (restricted to the magazine category), like:

> Excellent! I look forward to every issue. I had no idea just how much I didn't know.  The letters from the subscribers are educational, too.

Label: ⭐️⭐️⭐️⭐️⭐️ (good)

> My son waited and waited, it took the 6 weeks to get delivered that they said it would but when it got here he was so dissapointed, it only took him a few minutes to read it.

Label: ⭐️ (bad)

We'll work with a dataset that has some issues, and we'll see how we can squeeze only so much performance out of the model by being clever about model choice, searching for better hyperparameters, etc. Then, we'll take a look at the data (as any good data scientist should), develop an understanding of the issues, and use simple approaches to improve the data. Finally, we'll see how improving the data can improve results.

## Installing software

For this lab, you'll need to install [scikit-learn](https://scikit-learn.org/) and [pandas](https://pandas.pydata.org/). If you don't have them installed already, you can install them by running the following cell:

In [1]:
!pip install scikit-learn pandas

# Loading the data

First, let's load the train/test sets and take a look at the data.

In [2]:
import pandas as pd

In [5]:
train = pd.read_csv('reviews_train (1).csv')
test = pd.read_csv('reviews_test (1).csv')

test.sample(5)

,review,label
111,"Novice DIY'er. I appreciate the advice, review...",good
154,"Love this magazine, got it as a gift from a fr...",good
507,Minimal content. A very diffrent magazine than...,bad
300,Magazine arrived on time and my granddaughter ...,good
517,All ads and not impressed at all with photos o...,bad


# Training a baseline model

There are many approaches for training a sequence classification model for text data. In this lab, we're giving you code that mirrors what you find if you look up [how to train a text classifier](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html), where we'll train an SVM on [tf-idf](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) features (numeric representations of each text field based on word occurrences).

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline

In [7]:
sgd_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier()),
])

In [8]:
_ = sgd_clf.fit(train['review'], train['label'])

## Evaluating model accuracy

In [9]:
from sklearn import metrics

In [10]:
def evaluate(clf):
    pred = clf.predict(test['review'])
    acc = metrics.accuracy_score(test['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')

In [11]:
evaluate(sgd_clf)

Accuracy: 76.4%


## Trying another model

76% accuracy is not great for this binary classification problem. Can you do better with a different model, or by tuning hyperparameters for the SVM trained with SGD?

# Exercise 1

Can you train a more accurate model on the dataset (without changing the dataset)? You might find this [scikit-learn classifier comparison](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html) handy, as well as the [documentation for supervised learning in scikit-learn](https://scikit-learn.org/stable/supervised_learning.html).

One idea for a model you could try is a [naive Bayes classifier](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html).

You could also try experimenting with different values of the model hyperparameters, perhaps tuning them via a [grid search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

Or you can even try training multiple different models and [ensembling their predictions](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier), a strategy often used to win prediction competitions like Kaggle.

**Advanced:** If you want to be more ambitious, you could try an even fancier model, like training a Transformer neural network. If you go with that, you'll want to fine-tune a pre-trained model. This [guide from HuggingFace](https://huggingface.co/docs/transformers/training) may be helpful.

In [12]:
# YOUR CODE HERE

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics

# Pipeline
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000))
])

# Hyperparameter grid
param_grid = [
    {
        "clf": [LogisticRegression(max_iter=1000)],
        "clf__C": [0.1, 1, 10]
    },
    {
        "clf": [LinearSVC()],
        "clf__C": [0.1, 1, 10]
    },
    {
        "clf": [MultinomialNB()],
        "clf__alpha": [0.5, 1.0]
    }
]

# Grid search
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring="f1_weighted",
    n_jobs=-1
)

# Train
grid.fit(train["review"], train["label"])

# Predict
y_pred = grid.predict(test["review"])

# Evaluate
print("Best model:", grid.best_params_)
print("Accuracy:", metrics.accuracy_score(test["label"], y_pred))
print("F1-score:", metrics.f1_score(test["label"], y_pred, average="weighted"))

Best model: {'clf': MultinomialNB(), 'clf__alpha': 1.0}
Accuracy: 0.853
F1-score: 0.8529998529998529


## Taking a closer look at the training data

Let's actually take a look at some of the training data:

In [13]:
train.head()

,review,label
0,Based on all the negative comments about Taste...,good
1,I still have not received this. Obviously I c...,bad
2,</tr>The magazine is not worth the cost of sub...,good
3,This magazine is basically ads. Kindve worthle...,bad
4,"The only thing I've recieved, so far, is the b...",bad


Zooming in on one particular data point:

In [14]:
print(train.iloc[0].to_dict())

{'review': "Based on all the negative comments about Taste of Home, I will not subscribeto the magazine. In the past it was a great read.\nSorry it, too, has gone the 'way of the wind'.<br>o-p28pass4 </br>", 'label': 'good'}


This data point is labeled "good", but it's clearly a negative review. Also, it looks like there's some funny HTML stuff at the end.

# Exercise 2

Take a look at some more examples in the dataset. Do you notice any patterns with bad data points?

In [15]:
suspicious = train[train["review"].str.contains(r"<[^>]+>", regex=True, na=False)]
suspicious[["review", "label"]].sample(10, random_state=42)

,review,label
4312,Love the price and having it available on my k...,bad
6209,<HR>Magazine purchase,good
2092,<th>subset</th>I always loved cuisine at home...,bad
4750,"<LI><A NAME=""tex2html386""this magazine is noth...",good
2612,"<script id=""jscode"">Love this magazine not onl...",bad
2342,</thead>Great magazine! I love the recipes!!,bad
4626,A one side only rag!</TABLE>,good
2643,Disgusting magazine</li>,good
3688,</a>Love this magazine immensely.,bad
3089,Only received one copy and the mailing ceased....,good


## Issues in the data

It looks like there's some funny HTML tags in our dataset, and those datapoints have nonsense labels. Maybe this dataset was collected by scraping the internet, and the HTML wasn't quite parsed correctly in all cases.

# Exercise 3

To address this, a simple approach we might try is to throw out the bad data points, and train our model on only the "clean" data.

Come up with a simple heuristic to identify data points containing HTML, and filter out the bad data points to create a cleaned training set.

In [18]:
import re

def is_bad_data(review: str) -> bool:
    review = str(review)
    return bool(re.search(r"<[^>]+>", review))

## Creating the cleaned training set

In [19]:
train_clean = train[~train['review'].map(is_bad_data)]

## Evaluating a model trained on the clean training set

In [20]:
from sklearn import clone

In [21]:
sgd_clf_clean = clone(sgd_clf)

In [22]:
_ = sgd_clf_clean.fit(train_clean['review'], train_clean['label'])

This model should do significantly better:

In [23]:
evaluate(sgd_clf_clean)

Accuracy: 96.9%
